## 离线读者模式：构造滚动样本外配置

所有新计算值是教学构造方法演示，不是市场观测或历史结果复现。French原始、提取和处理数据不随包提供。G01数据权利、主体和用途尚未明确；A10-A仅有限设计接受，A10-A.1协议技术接受仍待许可与执行授权，G05/A10-B未授权。本Notebook不运行French，不运行P6再训练或第39章历史重建。保存输出若有是原文件携带记录，不是本轮新执行。


# Project 3：四种长仓组合的构造滚动比较

种子103生成2000-01起48个月、四资产A/B/C/D独立正态月度简单收益（均值0.005、标准差0.03），RF恒为0.001。调用项目共享核心，Spec窗口24、LW、滞后0，上限0.4仅约束最小方差与最大Sharpe，ERC无上限。持有2002-01至2003-12，共24个月/策略、完成时96行。它不读取冻结历史配置来运行French研究；默认写6个构造文件。完整历史440个月W60报告、练习与答案见项目导读，不与此构造W24混同。


In [ ]:
from pathlib import Path
import os, sys, uuid
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
for path in (ROOT / "src", ROOT / "code", ROOT):
    if str(path) not in sys.path: sys.path.insert(0, str(path))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("project03_portfolio_oos_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Run directory:", RUN_DIR.relative_to(ROOT))
import numpy as np, pandas as pd, json
import matplotlib.pyplot as plt
from IPython.display import display
sys.path.insert(0,str(ROOT / "projects/project03/src"))
from project03_backtest import STRATEGIES, Spec, run_strategy, apply_costs, metrics
from quant_course.performance import wealth_index
idx = pd.period_range("2000-01",periods=48,freq="M")
rng = np.random.default_rng(103)
returns = pd.DataFrame(rng.normal(0.005,0.03,(48,4)),index=idx,columns=list("ABCD"))
rf = pd.Series(0.001,index=idx)
spec = Spec(window=24,max_weight=0.4)
print("CONSTRUCTED OOS method; French branch SKIPPED: rights unknown")


## 1. 运行四策略的单个构造规格

24个月训练窗，10bp费用按权重换手代理扣除；此处不运行完整历史参数网格，也不承诺25秒耗时。


In [ ]:
ledgers = {}; weights = {}
for strategy in STRATEGIES:
    ledger, weights[strategy] = run_strategy(strategy,returns,rf,spec,"2002-01","2003-12")
    ledgers[strategy] = apply_costs(ledger,10.0)
primary = pd.DataFrame({s:metrics(l) for s,l in ledgers.items()}).T
display(primary)
primary.to_csv(RUN_DIR / "constructed_primary_metrics.csv")


## 2. 跳过历史summary比较

构造数据不可能复现French历史summary.json，下一格只说明跳过。


In [ ]:
print("Historical French summary comparison SKIPPED: this constructed panel cannot reproduce market results")


## 3. 账本恒等式与时点

首月从现金建仓，换手代理为1；之后先漂移再调仓。断言核对净收益与训练截止；四份账本写入项目RUN_DIR。


In [ ]:
led = ledgers["max_sharpe"]
assert np.allclose(led["net_return"], (1-led["cost"])*(1+led["gross_return"])-1)
assert (pd.PeriodIndex(led["train_end"],freq="M") < pd.PeriodIndex(led["holding_month"],freq="M")).all()
display(led[["holding_month","train_start","train_end","turnover","gross_return","cost","net_return"]].head())
for s,l in ledgers.items(): l.to_csv(RUN_DIR / (s+"_ledger_CONSTRUCTED.csv"),index=False)


## 4. 构造净财富

图含初始净值1，24个持有月共25点。原横轴Holding month是位置序号0至24，0在首月之前，不是日历日期；纵轴为线性净财富刻度。图片写入项目RUN_DIR。


In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
for s,l in ledgers.items(): ax.plot(wealth_index(l["net_return"].to_numpy()),label=s)
ax.set(title="CONSTRUCTED rolling allocation",xlabel="Holding month",ylabel="Net wealth"); ax.legend()
fig.tight_layout(); fig.savefig(RUN_DIR / "project03_wealth_CONSTRUCTED.png",dpi=130); plt.show()


## 5. 跳过历史块自助法与敏感性网格

下一格仅打印跳过说明，不计算Bootstrap区间。历史5000次/12月块结果见完整报告。


In [ ]:
print("Full historical bootstrap/sensitivity grid SKIPPED; the small constructed path checks timing and cost accounting only")


## 6. 本次构造路径的状态计数

末格按策略和状态汇总本次构造账本，不求后见之明组合。历史后见之明组合使用整个评价期信息、不可交易，仅在报告中作对照。


In [ ]:
print(pd.concat([l.assign(strategy=s) for s,l in ledgers.items()]).groupby(["strategy","status"]).size())
